<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="./img/btp-banner.gif" alt="BTP A&C">
</div>

# Tabular Prediction with SAP HANA Cloud's Built-In Procedure

This notebook demonstrates how to use SAP HANA Cloud's built-in **`AI_TABULAR_PREDICTION`** procedure to predict the delivery variance (in days) for purchase orders that are missing that value.

`AI_TABULAR_PREDICTION` is a HANA built-in procedure that delegates tabular prediction to an **RPT-1 (Relation Prediction Transformer)** foundation model deployed in SAP AI Core. The procedure is called inside a HANA **anonymous block** (`DO BEGIN ... END`) so that intermediate table variables can be constructed inline without creating permanent objects.

| Item | Detail |
|------|--------|
| Source table | `SPURCHASE.PO_HISTORY_V2` |
| Context | Random rows from `SPURCHASE.PO_HISTORY_V2` where `Delivery_Variance_Days` is known |
| Predict | All rows from `SPURCHASE.PO_HISTORY_V2` where `Delivery_Variance_Days` is `NULL` |
| Target column | `Delivery_Variance_Days` |
| Task type | `regression` |
| Remote source | `RPT_SMALL` |

## 🎯 Learning Objectives

- How to connect to SAP HANA Cloud using `hdbcli` and `python-dotenv`
- How to call `AI_TABULAR_PREDICTION` inside a HANA anonymous block using table variables
- How to interpret regression prediction results: `PREDICTED_VALUE` and `CONFIDENCE`

## 🚨 Prerequisites

Before running this notebook, complete the following setup:

1. **Create an RPT-1 deployment** in SAP AI Core:
   https://help.sap.com/docs/sap-ai-core/generative-ai/create-deployment-for-generative-ai-model

2. **Create a remote source** in SAP HANA Cloud pointing to the RPT-1 deployment:
   https://help.sap.com/docs/hana-cloud-database/sap-hana-cloud-sap-hana-database-data-access-guide/sap-hana-cloud-sap-hana-database-vector-engine-remote-source

3. **Grant EXECUTE privileges** on the remote source to your HANA user:
   ```sql
   GRANT EXECUTE ON REMOTE SOURCE <remote_source_name> TO <user_name>
   ```

4. Create a `.env` file in the project root containing:
   - `HANA_VECTOR_HOST` - Your HANA Cloud host URL (e.g. `xxx.hanacloud.ondemand.com`)
   - `HANA_VECTOR_USER` - Your HANA username
   - `HANA_VECTOR_PASS` - Your HANA password

---
## Step 1: Install and Import Required Libraries

| Library | Purpose |
|---------|----------|
| `hdbcli` | Low-level SAP HANA database driver used to execute SQL |
| `python-dotenv` | Loads credentials from a `.env` file |
| `pandas` | Formats the result set as a readable DataFrame |

> **Tip:** Run the install cell once, then restart the kernel before proceeding.

In [ ]:
%pip install hdbcli --quiet
%pip install python-dotenv --quiet
%pip install pandas --quiet
# Kernel restart may be required after first install

In [ ]:
import os
import pandas as pd
from hdbcli import dbapi
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

print('hdbcli version:', dbapi.apilevel)

---
## Step 2: Connect to SAP HANA Cloud

Credentials are read from environment variables so that sensitive values are never hard-coded in the notebook.

`dbapi.connect()` opens an SSL-encrypted connection to SAP HANA Cloud on port 443.

In [ ]:
HANA_HOST = os.getenv('HANA_VECTOR_HOST')
HANA_USER = os.getenv('HANA_VECTOR_USER')
HANA_PASS = os.getenv('HANA_VECTOR_PASS')

conn = dbapi.connect(
    address=HANA_HOST,
    port=443,
    user=HANA_USER,
    password=HANA_PASS
)

print(f'Connected to {HANA_HOST}:443 as {HANA_USER}')

---
## Step 3: Explore the Source Table

The single source table `SPURCHASE.PO_HISTORY_V2` contains all purchase order records:
- Rows where `Delivery_Variance_Days` **is not NULL** are used as **context** (known examples for the model).
- Rows where `Delivery_Variance_Days` **is NULL** are the rows to **predict**.

The anonymous block in the next step handles this split automatically using table variables.

In [ ]:
cursor = conn.cursor()

# Preview SPURCHASE.PO_HISTORY_V2
cursor.execute('SELECT TOP 5 * FROM SPURCHASE.PO_HISTORY_V2')
cols = [d[0] for d in cursor.description]
df_preview = pd.DataFrame(cursor.fetchall(), columns=cols)

print(f'Columns: {list(df_preview.columns)}')
df_preview

In [ ]:
# Row counts: known vs. to-predict
cursor.execute("""
    SELECT
        SUM(CASE WHEN "Delivery_Variance_Days" IS NOT NULL THEN 1 ELSE 0 END) AS KNOWN_ROWS,
        SUM(CASE WHEN "Delivery_Variance_Days" IS     NULL THEN 1 ELSE 0 END) AS ROWS_TO_PREDICT
    FROM SPURCHASE.PO_HISTORY_V2
""")
row = cursor.fetchone()
print(f'Rows with known Delivery_Variance_Days : {row[0]}')
print(f'Rows to predict (NULL)                 : {row[1]}')

---
## Step 4: Run AI_TABULAR_PREDICTION (Anonymous Block)

The prediction is executed inside a HANA **anonymous block** (`DO BEGIN ... END`). This allows the use of **table variables** to pass inline-constructed tables directly to `AI_TABULAR_PREDICTION` without creating any permanent objects.

The block:
1. Populates `DATA_WITH_MISSING_VALUES` with the rows to score (those where `Delivery_Variance_Days` is `NULL`).
2. Populates `CONTEXT` with the randomly sampled rows that have a known `Delivery_Variance_Days` value.
3. Calls `AI_TABULAR_PREDICTION` with a **regression** configuration targeting `Delivery_Variance_Days`.
4. Returns the predictions with `SELECT * FROM :"PREDICTED_VALUES"`.

The result set (one row per predicted value) contains:

| Column | Description |
|--------|-------------|
| `ID` | `PO_ID` of the input row (as string) |
| `COLUMN_NAME` | `Delivery_Variance_Days` |
| `PREDICTED_VALUE` | Predicted number of days variance (as string) |
| `CONFIDENCE` | Model confidence score |

In [ ]:
sql = """
DO
BEGIN
    "DATA_WITH_MISSING_VALUES" = 
        SELECT "PO_ID","Vendor","Vendor_Country","Vendor_OTIF_Percent",
               "Vendor_Avg_Past_Delay","Material_ID","Material_Description",
               "Material_Group","Criticality_Flag","Plant_ID","Purchasing_Group",
               "Order_Quantity","Net_Price","Planned_Lead_Time_Days",
               "Order_Month","Incoterms"
        FROM "SPURCHASE"."PO_HISTORY_V2"
        WHERE "Delivery_Variance_Days" IS NULL;

    "CONTEXT" = 
        SELECT "PO_ID","Vendor","Vendor_Country","Vendor_OTIF_Percent",
               "Vendor_Avg_Past_Delay","Material_ID","Material_Description",
               "Material_Group","Criticality_Flag","Plant_ID","Purchasing_Group",
               "Order_Quantity","Net_Price","Planned_Lead_Time_Days",
               "Order_Month","Incoterms","Delivery_Variance_Days"
        FROM "SPURCHASE"."PO_HISTORY_V2"
        ORDER BY RAND() LIMIT 1000;

    CALL AI_TABULAR_PREDICTION (
        REMOTE_SOURCE_NAME => 'RPT_SMALL',
        PREDICT_TABLE      => :"DATA_WITH_MISSING_VALUES",
        CONTEXT_TABLE      => :"CONTEXT",
        ID_COLUMN_NAME     => 'PO_ID',
        CONFIGURATION      => '{"target_columns":[{"name":"Delivery_Variance_Days","task_type":"regression"}]}',
        PREDICT_RESULTS    => "PREDICTED_VALUES"
    );

    SELECT * FROM :"PREDICTED_VALUES";
END
"""

print('Calling AI_TABULAR_PREDICTION ...')
cursor.execute(sql)
print('Done.')

---
## Step 5: Display Prediction Results

The `SELECT * FROM :"PREDICTED_VALUES"` at the end of the anonymous block produces a result set that is returned directly to the Python cursor.

- **`PREDICTED_VALUE`** - the model's predicted `Delivery_Variance_Days` (returned as a string; cast to `float` for numeric use).
- **`CONFIDENCE`** - the model's confidence in the prediction (0.0 to 1.0).

In [ ]:
# Fetch and display the raw prediction results
rows    = cursor.fetchall()
columns = [d[0] for d in cursor.description]

df_results = pd.DataFrame(rows, columns=columns)

# Cast PREDICTED_VALUE to float for numeric analysis
df_results['PREDICTED_VALUE'] = pd.to_numeric(df_results['PREDICTED_VALUE'], errors='coerce')

print(f'Predictions returned: {len(df_results)}')
df_results

In [ ]:
# Summary statistics of the predicted delivery variance
df_results['PREDICTED_VALUE'].describe()

---
## Step 6: Close Connection

Always close the HANA connection when work is complete to release the database session and free up connection-pool resources.

In [ ]:
cursor.close()
conn.close()
print('Connection closed.')